In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from processL0b import utils
from processL0b.reader import Reader

## Group 1
For each radiometer row, we want to associate a temperature row and a gps row.
Find the closest system timestamp and tie those together.

In [ ]:
# This gets the radiometer rows between the ith and ith + 1 thermistor timestamp
# Each step contains about 5500 rows, but some rows contain double or even triple? Why is that?
# for i in range(len(r.thermistors.data) - 1):
r = Reader("C:\\Users\\agwhi\\Desktop\\cristal\\data413\\l0b\\26_04_13__13_29_44__cristalANT.h5")
i = 3
r.radiometer.data[
    (r.radiometer.data['Timestamp'] - r.thermistors.temps['Timestamp'][i] > 0)
    &
    (r.radiometer.data['Timestamp'] - r.thermistors.temps['Timestamp'][i+1] < 0)
]

In [ ]:
# Get rows at nadir for each revolution
MOTOR_NADIR = [3500, 4000]

nadir_per_revolution = r.radiometer.data[
    (r.radiometer.data['MotorPosition'] > MOTOR_NADIR[0])
    &
    (r.radiometer.data['MotorPosition'] < MOTOR_NADIR[1])
].groupby('Revolution').mean()
nadir_per_revolution

In [ ]:
# Average a row per revolution between the given values
MOTOR_ZENITH = [11500, 12000]

zenith_per_revolution = r.radiometer.data[
    (r.radiometer.data['MotorPosition'] > MOTOR_ZENITH[0])
    &
    (r.radiometer.data['MotorPosition'] < MOTOR_ZENITH[1])
].groupby('Revolution').mean()

CALTAR_THERMISTORS = [9, 10, 11, 12, 13, 14, 15, 16]

def find_closest_index(series, value):
    return series.iloc[
        (series - value).abs().argsort()[:1]
    ].index[0]

i = []
for timestamp in zenith_per_revolution['Timestamp']:
    i.append( find_closest_index(r.thermistors.data['Timestamp'], timestamp) )
means = r.thermistors.data.iloc[i][CALTAR_THERMISTORS].mean(axis=1).reset_index(drop=True)

zenith_per_revolution['Temps'] = means
zenith_per_revolution

In [ ]:
# For each channel, gain and offset are determined
LN2_TEMP = 77
labels = [ch.label for ch in r.radiometer.channels]
df = pd.DataFrame({
    'zenith': zenith_per_revolution[labels].mean(),
    'nadir': nadir_per_revolution[labels].mean(),
})
df['gain'] = (df['zenith'] - df['nadir']) / (zenith_per_revolution['Temps'].mean() - LN2_TEMP)
df['offset'] = df['zenith'] - df['gain'] * zenith_per_revolution['Temps'].mean()
# df = df.transpose()
r.radiometer.data['34 QV'] * df.loc['34 QV', 'gain']
# tb = (r.radiometer.data[labels] - df.loc['offset']) / df.loc['gain']

# for ch in r.radiometer.channels:
#     if ch.frequency == 0:
#         continue
#     plt.plot(
#         r.radiometer.data['MotorPosition'], tb[ch.label], linestyle='none', marker='.',
#         label=ch.label,
#     )
# plt.legend()
# plt.show()


## Group 2
Replicate the motor angle vs revolution brightness plot

In [2]:
def apply_calibration_values(calibration: pd.DataFrame, l0b_filename: str) -> pd.DataFrame:
    reader = Reader(l0b_filename)
    brightness_temps = reader.radiometer.data
    for channel in reader.radiometer.channels:
        brightness_temps[channel.label] = (brightness_temps[channel.label] - calibration.loc[channel.label, 'Offset']) / calibration.loc[channel.label, 'Gain']
    return brightness_temps

df = apply_calibration_values(
    utils.load_cal_file("../calfile.csv"),
    "C:\\Users\\agwhi\\Desktop\\cristal\\data413\\l0b\\26_04_13__13_29_44__cristalANT.h5",
)
df.head()

c:\Users\agwhi\Desktop\Code\DataSystemPy3\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,Timestamp,MotorPosition,34 QV,Not Connected 1,18 QV,24 QV,34 QH,Not Connected 2,18 QH,24 QH,Revolution
0,1.776109e+09,11599,197.319645,-5455.492661,376.819048,252.327493,249.573183,-8397.073132,386.993489,256.372087,0
1,1.776109e+09,11602,201.976908,-8554.858483,350.811740,259.667860,251.754683,-7133.998673,390.544021,252.812881,0
2,1.776109e+09,11605,213.082688,-13203.907216,353.585853,260.531433,250.118558,-8397.073132,382.213925,255.886741,0
3,1.776109e+09,11608,195.170139,-8554.858483,356.533348,266.576441,251.209308,-9660.147590,373.474153,258.960601,0
4,1.776109e+09,11611,200.902155,-806.443929,361.908191,255.997677,243.028682,-7133.998673,381.121454,256.695652,0


In [11]:
%matplotlib qt6
data = df[['Revolution', 'MotorPosition', '34 QV']][:2**16]
x = data['Revolution']
y = data['MotorPosition'] * (360/16000)
z = data['34 QV']

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(x, y, z)